# Balanced GELU UniversalXAS and Tuned-UniversalXAS FEFF

This notebook trains one current best-setting GELU UniversalXAS model and one fixed GELU Tuned-UniversalXAS model per FEFF dataset. It uses exported 64D features and does not retrain the encoder.

> **Exploratory combined change.** This run combines GELU activation with the balanced loader. Universal results cannot isolate activation effects from loader effects. Tuned models also change the source weights and activation relative to the prior balanced run.

Tuned settings come from the prior run's validation-only candidate file. Test eta never selects a setting. The notebook writes all requested artifacts directly under `SOURCE_RUN/train_balanced_gelu_universal_tuned_feff`.


## 1. Imports and fixed configuration

The source run comes from `OMNIXAS_E2E_UNIVERSAL_RUN`. The fallback path is the fixed exported-feature run under `fulltrainingcopy072726`.


In [1]:
from __future__ import annotations

from collections.abc import Iterator
from dataclasses import asdict, dataclass
from hashlib import sha256
import json
import math
import os
from pathlib import Path
from typing import Any

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import lightning.pytorch as pl
import numpy as np
import pandas as pd
import torch
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import DataLoader, Sampler, TensorDataset

from omnixas.data.ml_data import MLData, MLSplits
from omnixas.model.training import PlModule

torch.set_float32_matmul_precision("medium")

ELEMENTS = ["Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu"]
FEFF_DATASETS = [f"{element}_FEFF" for element in ELEMENTS]
INPUT_DIM, OUTPUT_DIM = 64, 141
HIDDEN_DIMS = [500, 500, 550]
UNIVERSAL_SEED = 44
UNIVERSAL_DROPOUT = 0.10
UNIVERSAL_LR = 5e-4
UNIVERSAL_BATCH_SIZE = 32
UNIVERSAL_ROWS_PER_ELEMENT = 4
UNIVERSAL_SCHEDULER = "ReduceLROnPlateau"
UNIVERSAL_SCHEDULER_MODE = "min"
UNIVERSAL_SCHEDULER_FACTOR = 0.5
UNIVERSAL_BASE_SCHEDULER_PATIENCE = 8
UNIVERSAL_SCHEDULER_FREQUENCY = 2
UNIVERSAL_MIN_LR = 1e-6
UNIVERSAL_MAX_EPOCHS = 800
UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE = 60
CHECK_VAL_EVERY_N_EPOCH = 2
UNIVERSAL_MONITOR = "val_median_mse"
TUNED_MONITOR = "val_median_mse"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def locate_repo_root(start: Path) -> Path:
    configured = os.environ.get("OMNIXAS_REPO_ROOT")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if (candidate / "pyproject.toml").is_file() and (candidate / "omnixas").is_dir():
            return candidate
        raise FileNotFoundError(f"OMNIXAS_REPO_ROOT is not an OmniXAS repository: {candidate}")
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "omnixas").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the OmniXAS repository")


def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def json_value(value: Any) -> Any:
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_value(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_value(item) for item in value]
    return value


def fingerprint(value: Any) -> str:
    payload = json.dumps(json_value(value), sort_keys=True, separators=(",", ":"))
    return sha256(payload.encode("utf-8")).hexdigest()


REPO_ROOT = locate_repo_root(Path.cwd().resolve())
SOURCE_RUN = Path(
    os.environ.get(
        "OMNIXAS_E2E_UNIVERSAL_RUN",
        REPO_ROOT.parent / "fulltrainingcopy072726" / "m3gnetAll8E2EUniversal" / "e2e_universal_seed42",
    )
).expanduser().resolve()
FEATURES_DIR = SOURCE_RUN / "features"
ML_DATA_DIR = REPO_ROOT / "tutorial_omnixas" / "ml_data"
MATERIAL_IDS_DIR = REPO_ROOT / "tutorial_omnixas" / "material_id_and_site"
PRIOR_RUN = SOURCE_RUN / "train_balanced_e2e_universal_tuned_feff"
PRIOR_VALIDATION_CSV = PRIOR_RUN / "analysis" / "tuned_validation_candidates.csv"
RUN_DIR = SOURCE_RUN / "train_balanced_gelu_universal_tuned_feff"

for required in (SOURCE_RUN, FEATURES_DIR, ML_DATA_DIR, MATERIAL_IDS_DIR, PRIOR_VALIDATION_CSV):
    if not required.exists():
        raise FileNotFoundError(f"Missing required path: {required}")

print(f"Source run: {SOURCE_RUN}")
print(f"Prior validation candidates: {PRIOR_VALIDATION_CSV}")
print(f"Output run: {RUN_DIR}")
print(f"Device: {DEVICE}")


Source run: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42
Prior validation candidates: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/train_balanced_e2e_universal_tuned_feff/analysis/tuned_validation_candidates.csv
Output run: /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/train_balanced_gelu_universal_tuned_feff
Device: cuda:0


## 2. Load and validate fixed exported data

The exported feature files supply `X`. The exported `y` files must match the canonical targets in `tutorial_omnixas/ml_data` exactly, row by row. The material/site files must match row counts, contain no duplicate full IDs, and keep material IDs isolated across train, validation, and test.


In [2]:
@dataclass(frozen=True)
class LoadedSplit:
    split: MLSplits
    ids: dict[str, list[str]]
    materials: dict[str, list[str]]


def read_matrix(path: Path) -> np.ndarray:
    return np.atleast_2d(np.loadtxt(path, dtype=np.float32))


def read_ids(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def load_dataset(dataset: str) -> LoadedSplit:
    parts: dict[str, MLData] = {}
    ids_by_split: dict[str, list[str]] = {}
    materials_by_split: dict[str, list[str]] = {}
    seen_ids: set[str] = set()
    for split_name in ("train", "val", "test"):
        x_path = FEATURES_DIR / f"{dataset}_{split_name}_X.txt"
        exported_y_path = FEATURES_DIR / f"{dataset}_{split_name}_y.txt"
        canonical_y_path = ML_DATA_DIR / f"{dataset}_{split_name}_y.txt"
        ids_path = MATERIAL_IDS_DIR / f"{dataset}_{split_name}.txt"
        for path in (x_path, exported_y_path, canonical_y_path, ids_path):
            if not path.is_file():
                raise FileNotFoundError(f"Missing {dataset} {split_name} file: {path}")

        X = read_matrix(x_path)
        exported_y = read_matrix(exported_y_path)
        canonical_y = read_matrix(canonical_y_path)
        if X.shape != (X.shape[0], INPUT_DIM) or exported_y.shape != (exported_y.shape[0], OUTPUT_DIM):
            raise ValueError(f"Invalid exported dimensions for {dataset} {split_name}: X={X.shape}, y={exported_y.shape}")
        if canonical_y.shape != exported_y.shape or X.shape[0] != exported_y.shape[0]:
            raise ValueError(f"Row or target dimensions do not match for {dataset} {split_name}")
        if not np.isfinite(X).all() or not np.isfinite(exported_y).all() or not np.isfinite(canonical_y).all():
            raise ValueError(f"Non-finite data in {dataset} {split_name}")
        if not np.array_equal(exported_y, canonical_y):
            raise ValueError(f"Exported targets do not exactly match tutorial_omnixas/ml_data for {dataset} {split_name}")

        ids = read_ids(ids_path)
        if len(ids) != X.shape[0]:
            raise ValueError(f"ID row count mismatch for {dataset} {split_name}: IDs={len(ids)}, rows={X.shape[0]}")
        if len(set(ids)) != len(ids):
            raise ValueError(f"Duplicate full material/site IDs in {ids_path}")
        overlap = seen_ids.intersection(ids)
        if overlap:
            raise ValueError(f"Duplicate full IDs across splits for {dataset}: {sorted(overlap)[:5]}")
        seen_ids.update(ids)
        materials = []
        for row in ids:
            material, separator, site = row.rpartition("_")
            if not separator or not material or not site or not site.isascii() or not site.isdigit() or int(site) < 0:
                raise ValueError(f"Invalid material/site ID with nonnegative integer site suffix in {ids_path}: {row!r}")
            materials.append(material)
        ids_by_split[split_name] = ids
        materials_by_split[split_name] = materials
        parts[split_name] = MLData(X=X, y=exported_y)

    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        overlap = set(materials_by_split[left]).intersection(materials_by_split[right])
        if overlap:
            raise ValueError(f"Material split isolation failed for {dataset} {left}/{right}: {sorted(overlap)[:5]}")
    return LoadedSplit(MLSplits(**parts), ids_by_split, materials_by_split)


loaded = {dataset: load_dataset(dataset) for dataset in FEFF_DATASETS}
splits = {dataset: item.split for dataset, item in loaded.items()}
source_files = []
for dataset in FEFF_DATASETS:
    for split_name in ("train", "val", "test"):
        source_files.extend(
            [
                FEATURES_DIR / f"{dataset}_{split_name}_X.txt",
                FEATURES_DIR / f"{dataset}_{split_name}_y.txt",
                ML_DATA_DIR / f"{dataset}_{split_name}_y.txt",
                MATERIAL_IDS_DIR / f"{dataset}_{split_name}.txt",
            ]
        )
source_files = sorted(set(source_files))
print({dataset: {name: len(getattr(split, name)) for name in ("train", "val", "test")} for dataset, split in splits.items()})


{'Ti_FEFF': {'train': 5140, 'val': 641, 'test': 641}, 'V_FEFF': {'train': 8653, 'val': 1080, 'test': 1080}, 'Cr_FEFF': {'train': 2457, 'val': 305, 'test': 305}, 'Mn_FEFF': {'train': 13644, 'val': 1704, 'test': 1704}, 'Fe_FEFF': {'train': 9657, 'val': 1205, 'test': 1205}, 'Co_FEFF': {'train': 8605, 'val': 1074, 'test': 1074}, 'Ni_FEFF': {'train': 3471, 'val': 432, 'test': 432}, 'Cu_FEFF': {'train': 3340, 'val': 416, 'test': 416}}


## 3. GELU model, balanced sampler, and budget self-check

The local model has `64 -> 500 -> 500 -> 550 -> 141`. Every hidden layer uses `Linear`, `BatchNorm1d`, `GELU`, and `Dropout`. The output uses `Softplus`.

The sampler redraws without replacement on every epoch with `seed + epoch`. The separate self-check sampler runs before the real training sampler is created.


In [3]:
class GELUXASBlock(nn.Sequential):
    def __init__(self, input_dim: int, hidden_dims: list[int], output_dim: int, dropout: float):
        dims = [input_dim, *hidden_dims, output_dim]
        layers: list[nn.Module] = []
        for index, (width_in, width_out) in enumerate(zip(dims[:-1], dims[1:])):
            layers.append(nn.Linear(width_in, width_out))
            if index < len(dims) - 2:
                layers.extend([nn.BatchNorm1d(width_out), nn.GELU(), nn.Dropout(dropout)])
            else:
                layers.append(nn.Softplus())
        super().__init__(*layers)


class ElementStratifiedBatchSampler(Sampler[list[int]]):
    def __init__(self, indices_by_element: dict[int, list[int]], per_element: int, seed: int):
        if not indices_by_element or set(indices_by_element) != set(range(len(ELEMENTS))):
            raise ValueError("Sampler requires one non-empty pool for each of the eight elements")
        if per_element < 1:
            raise ValueError("per_element must be positive")
        self.indices_by_element = {int(key): list(value) for key, value in indices_by_element.items()}
        self.element_order = sorted(self.indices_by_element)
        self.per_element = int(per_element)
        self.seed = int(seed)
        self.epoch = 0
        counts = [len(self.indices_by_element[key]) for key in self.element_order]
        self.n_batches = min(counts) // self.per_element
        if self.n_batches < 1:
            raise ValueError("Every element needs at least per_element rows")

    def __len__(self) -> int:
        return self.n_batches

    def __iter__(self) -> Iterator[list[int]]:
        rng = np.random.default_rng(self.seed + self.epoch)
        self.epoch += 1
        selected = {
            element: rng.permutation(self.indices_by_element[element])[: self.n_batches * self.per_element]
            for element in self.element_order
        }
        batches = []
        for batch_number in range(self.n_batches):
            batch = np.concatenate(
                [
                    selected[element][batch_number * self.per_element : (batch_number + 1) * self.per_element]
                    for element in self.element_order
                ]
            )
            rng.shuffle(batch)
            batches.append(batch.tolist())
        return iter(batches)


def combine_split(split_name: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    X_blocks, y_blocks, element_blocks = [], [], []
    for element_index, dataset in enumerate(FEFF_DATASETS):
        part = getattr(splits[dataset], split_name)
        X_blocks.append(part.X)
        y_blocks.append(part.y)
        element_blocks.append(np.full(len(part), element_index, dtype=np.int64))
    return np.concatenate(X_blocks), np.concatenate(y_blocks), np.concatenate(element_blocks)


universal_train_X, universal_train_y, train_element_codes = combine_split("train")
universal_val_X, universal_val_y, val_element_codes = combine_split("val")
train_indices_by_element = {
    element_index: np.flatnonzero(train_element_codes == element_index).tolist()
    for element_index in range(len(ELEMENTS))
}
val_baselines = np.array(
    [
        np.median(np.mean((splits[dataset].val.y - splits[dataset].train.y.mean(axis=0)) ** 2, axis=1))
        for dataset in FEFF_DATASETS
    ],
    dtype=np.float32,
)

self_check_sampler = ElementStratifiedBatchSampler(train_indices_by_element, UNIVERSAL_ROWS_PER_ELEMENT, UNIVERSAL_SEED)
expected_batches = min(len(pool) for pool in train_indices_by_element.values()) // UNIVERSAL_ROWS_PER_ELEMENT
self_check_batches = list(self_check_sampler)
if len(self_check_batches) != expected_batches:
    raise AssertionError(f"Sampler batch count mismatch: {len(self_check_batches)} != {expected_batches}")
all_sampled_indices = []
for batch in self_check_batches:
    if len(batch) != UNIVERSAL_BATCH_SIZE:
        raise AssertionError(f"Sampler batch size mismatch: {len(batch)}")
    counts = np.bincount(train_element_codes[np.asarray(batch)], minlength=len(ELEMENTS))
    if not np.array_equal(counts, np.full(len(ELEMENTS), UNIVERSAL_ROWS_PER_ELEMENT)):
        raise AssertionError(f"Sampler element counts are not exactly four: {counts.tolist()}")
    all_sampled_indices.extend(batch)
if len(all_sampled_indices) != len(set(all_sampled_indices)):
    raise AssertionError("Sampler repeated a row index within one epoch")

real_sampler = ElementStratifiedBatchSampler(train_indices_by_element, UNIVERSAL_ROWS_PER_ELEMENT, UNIVERSAL_SEED)
full_data_batches = math.ceil(len(universal_train_y) / UNIVERSAL_BATCH_SIZE)
balanced_batches = len(real_sampler)
update_budget_scale = full_data_batches / balanced_batches
effective_scheduler_patience = round(UNIVERSAL_BASE_SCHEDULER_PATIENCE * update_budget_scale)


In [4]:
effective_early_stopping_patience = round(UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE * update_budget_scale)
if (effective_scheduler_patience, effective_early_stopping_patience) != (22, 168):
    raise ValueError(
        "Current data does not produce the established effective patience values "
        f"(22, 168): got ({effective_scheduler_patience}, {effective_early_stopping_patience})"
    )
print(
    f"Sampler self-check passed: {expected_batches} batches, "
    f"full={full_data_batches}, balanced={balanced_batches}, "
    f"scale={update_budget_scale:.6f}, patience={effective_scheduler_patience}/{effective_early_stopping_patience}"
)


Sampler self-check passed: 614 batches, full=1718, balanced=614, scale=2.798046, patience=22/168


## 4. Prior validation-only tuned settings and safe run state

This cell reads only the prior validation candidate file. It selects the unique maximum `val_eta` row per task. It does not read prior test metrics. The selected values transfer hyperparameters only. The old SiLU weights are never loaded.

A non-empty output directory must contain a matching manifest. Every setting directory must contain matching settings, a valid `TRAINING_COMPLETE.json`, and exactly one `best*.ckpt` before reuse.


In [5]:
prior_candidates = pd.read_csv(PRIOR_VALIDATION_CSV)
required_candidate_columns = {
    "dataset", "val_eta", "seed", "dropout", "lr", "cosine_t", "eta_min", "patience", "task_batch", "setting"
}
missing = required_candidate_columns.difference(prior_candidates.columns)
epoch_column = "max_epochs" if "max_epochs" in prior_candidates.columns else "epochs"
if missing or epoch_column not in prior_candidates.columns:
    raise ValueError(f"Prior validation CSV is missing required columns: {sorted(missing | ({epoch_column} if epoch_column not in prior_candidates.columns else set()))}")
if set(prior_candidates["dataset"]) != set(FEFF_DATASETS):
    raise ValueError("Prior validation candidates must contain exactly the eight FEFF tasks")
if not np.isfinite(prior_candidates["val_eta"].to_numpy(dtype=float)).all():
    raise ValueError("Prior validation candidates contain non-finite val_eta")

selected_prior_rows: dict[str, dict[str, Any]] = {}
tuned_configs: dict[str, dict[str, Any]] = {}
for dataset in FEFF_DATASETS:
    task_rows = prior_candidates[prior_candidates["dataset"] == dataset]
    if len(task_rows) < 1:
        raise ValueError(f"No prior validation candidates for {dataset}")
    max_eta = task_rows["val_eta"].max()
    best_rows = task_rows[task_rows["val_eta"] == max_eta]
    if len(best_rows) != 1:
        raise ValueError(f"Prior validation selection is not unique for {dataset}: {len(best_rows)} rows tie for max val_eta")
    row = best_rows.iloc[0].to_dict()
    selected_prior_rows[dataset] = json_value(row)
    tuned_configs[dataset] = {
        "setting": str(row["setting"]),
        "seed": int(row["seed"]),
        "dropout": float(row["dropout"]),
        "lr": float(row["lr"]),
        "cosine_t": int(row["cosine_t"]),
        "eta_min": float(row["eta_min"]),
        "max_epochs": int(row[epoch_column]),
        "patience": int(row["patience"]),
        "task_batch": int(row["task_batch"]),
    }
    if not (0 < tuned_configs[dataset]["dropout"] < 1 and tuned_configs[dataset]["lr"] > 0 and tuned_configs[dataset]["eta_min"] > 0):
        raise ValueError(f"Invalid selected tuned settings for {dataset}")
if len(tuned_configs) != len(FEFF_DATASETS):
    raise AssertionError("Expected one selected tuned configuration per FEFF task")
print(pd.DataFrame(tuned_configs).T[["setting", "seed", "dropout", "lr", "cosine_t", "eta_min", "max_epochs", "patience", "task_batch"]])

universal_settings = {
    "model": "UniversalXAS",
    "activation": "GELU",
    "input_dim": INPUT_DIM,
    "hidden_dims": HIDDEN_DIMS,
    "output_dim": OUTPUT_DIM,
    "seed": UNIVERSAL_SEED,
    "dropout": UNIVERSAL_DROPOUT,
    "optimizer": "Adam",
    "lr": UNIVERSAL_LR,
    "batch_size": UNIVERSAL_BATCH_SIZE,
    "rows_per_element": UNIVERSAL_ROWS_PER_ELEMENT,
    "sampler": "ElementStratifiedBatchSampler",
    "scheduler": UNIVERSAL_SCHEDULER,
    "scheduler_mode": UNIVERSAL_SCHEDULER_MODE,
    "scheduler_factor": UNIVERSAL_SCHEDULER_FACTOR,
    "scheduler_base_patience": UNIVERSAL_BASE_SCHEDULER_PATIENCE,
    "scheduler_effective_patience": effective_scheduler_patience,
    "scheduler_frequency": UNIVERSAL_SCHEDULER_FREQUENCY,
    "min_lr": UNIVERSAL_MIN_LR,
    "max_epochs": UNIVERSAL_MAX_EPOCHS,
    "early_stopping_base_patience": UNIVERSAL_BASE_EARLY_STOPPING_PATIENCE,
    "early_stopping_effective_patience": effective_early_stopping_patience,
    "monitor": UNIVERSAL_MONITOR,
    "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
}
settings_core = {
    "source_run": str(SOURCE_RUN),
    "feature_provenance": "fixed exported 64D OMNIXAS_E2E_UNIVERSAL_RUN features",
    "encoder_retrained": False,
    "datasets": FEFF_DATASETS,
    "universal": universal_settings,
    "tuned_configs": tuned_configs,
    "prior_validation_csv": str(PRIOR_VALIDATION_CSV),
    "prior_validation_selected_rows": selected_prior_rows,
}
settings_fingerprint = fingerprint(settings_core)
manifest_path = RUN_DIR / "settings_manifest.json"
if RUN_DIR.exists():
    if not RUN_DIR.is_dir():
        raise FileExistsError(f"Output path is not a directory: {RUN_DIR}")
    if any(RUN_DIR.iterdir()):
        if not manifest_path.is_file():
            raise RuntimeError(f"Non-empty run directory has no settings_manifest.json: {RUN_DIR}")
        existing = json.loads(manifest_path.read_text(encoding="utf-8"))
        if existing.get("configuration_fingerprint") != settings_fingerprint:
            raise RuntimeError("Existing run settings do not match this notebook")
        print(f"Validated existing run directory: {RUN_DIR}")
else:
    RUN_DIR.mkdir(parents=True)

manifest = {
    **settings_core,
    "configuration_fingerprint": settings_fingerprint,
    "status": "incomplete",
    "output_run": str(RUN_DIR),
    "source_files": [{"path": str(path), "sha256": sha256_file(path)} for path in source_files],
    "prior_run": str(PRIOR_RUN),
    "prior_validation_source": str(PRIOR_VALIDATION_CSV),
    "universal_full_data_batches_per_epoch": full_data_batches,
    "universal_balanced_batches_per_epoch": balanced_batches,
    "universal_update_budget_scale": update_budget_scale,
    "effective_scheduler_patience": effective_scheduler_patience,
    "effective_early_stopping_patience": effective_early_stopping_patience,
    "sampler_details": {
        "batch_size": UNIVERSAL_BATCH_SIZE,
        "rows_per_element": UNIVERSAL_ROWS_PER_ELEMENT,
        "element_codes": {str(index): element for index, element in enumerate(ELEMENTS)},
        "redraw": "without replacement per element pool with seed + epoch",
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")


                           setting seed dropout      lr cosine_t   eta_min  \
Ti_FEFF   cosT750_lr3e-4_do0p1_p60  145     0.1  0.0003      750  0.000001   
V_FEFF   cosT500_lr3e-4_do0p15_p60  145    0.15  0.0003      500  0.000001   
Cr_FEFF  cosT500_lr3e-4_do0p15_p60  145    0.15  0.0003      500  0.000001   
Mn_FEFF   cosT750_lr4e-4_do0p1_p60  145     0.1  0.0004      750  0.000001   
Fe_FEFF  cosT500_lr3e-4_do0p15_p60  145    0.15  0.0003      500  0.000001   
Co_FEFF  cosT500_lr3e-4_do0p05_p60  145    0.05  0.0003      500  0.000001   
Ni_FEFF   cosT500_lr3e-4_do0p1_p60  145     0.1  0.0003      500  0.000001   
Cu_FEFF  cosT500_lr3e-4_do0p15_p60  145    0.15  0.0003      500  0.000001   

        max_epochs patience task_batch  
Ti_FEFF       1000       60         32  
V_FEFF        1000       60         32  
Cr_FEFF       1000       60         32  
Mn_FEFF       1000       60         64  
Fe_FEFF       1000       60         64  
Co_FEFF       1000       60         32  
Ni_FEFF   

39475

## 5. Local training and checkpoint helpers

Both stages use local GELU modules. Checkpoints monitor `val_median_mse`, save one best checkpoint, save the last checkpoint, use CSV logs, and show the progress bar.


In [6]:
class UniversalBalancedData(pl.LightningDataModule):
    def __init__(self, sampler: ElementStratifiedBatchSampler):
        super().__init__()
        self.train = TensorDataset(
            torch.as_tensor(universal_train_X, dtype=torch.float32),
            torch.as_tensor(universal_train_y, dtype=torch.float32),
            torch.as_tensor(train_element_codes, dtype=torch.long),
        )
        self.val = TensorDataset(
            torch.as_tensor(universal_val_X, dtype=torch.float32),
            torch.as_tensor(universal_val_y, dtype=torch.float32),
            torch.as_tensor(val_element_codes, dtype=torch.long),
        )
        self.sampler = sampler

    def train_dataloader(self):
        return DataLoader(self.train, batch_sampler=self.sampler)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=1024, shuffle=False)


class StandardTaskData(pl.LightningDataModule):
    def __init__(self, split: MLSplits, batch_size: int):
        super().__init__()
        self.train = TensorDataset(torch.as_tensor(split.train.X), torch.as_tensor(split.train.y))
        self.val = TensorDataset(torch.as_tensor(split.val.X), torch.as_tensor(split.val.y))
        self.batch_size = batch_size

    def train_dataloader(self):
        return DataLoader(self.train, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val, batch_size=1024, shuffle=False)


class BalancedUniversalModule(PlModule):
    def __init__(self, val_baselines: np.ndarray, **kwargs):
        super().__init__(**kwargs)
        self.register_buffer("val_baselines", torch.as_tensor(val_baselines, dtype=torch.float32))
        self.val_mses: list[torch.Tensor] = []
        self.val_elements: list[torch.Tensor] = []

    def training_step(self, batch, batch_idx):
        x, y, _ = batch
        return self.logged_loss("train_loss", y, self.model(x))

    def on_validation_epoch_start(self):
        self.val_mses, self.val_elements = [], []

    def validation_step(self, batch, batch_idx):
        x, y, elements = batch
        y_pred = self.model(x)
        self.val_mses.append(torch.mean((y - y_pred) ** 2, dim=1).detach())
        self.val_elements.append(elements.detach())
        return self.logged_loss("val_loss", y, y_pred)

    def on_validation_epoch_end(self):
        if self.trainer.sanity_checking:
            return
        if not self.val_mses:
            raise RuntimeError("Validation produced no rows")
        mses = torch.cat(self.val_mses)
        elements = torch.cat(self.val_elements)
        self.log("val_median_mse", torch.quantile(mses, 0.5), on_step=False, on_epoch=True, prog_bar=True)
        relative_medians = []
        for element_index, element in enumerate(ELEMENTS):
            mask = elements == element_index
            if not mask.any():
                raise RuntimeError(f"Validation has no rows for {element}")
            relative_medians.append(torch.quantile(mses[mask], 0.5) / self.val_baselines[element_index])
        self.log("val_balanced_rel_mse", torch.stack(relative_medians).mean(), on_step=False, on_epoch=True, prog_bar=True)


TRAINING_COMPLETE_FILENAME = "TRAINING_COMPLETE.json"


def _validate_setting_dir(setting_dir: Path) -> Path:
    setting_dir = setting_dir.resolve()
    run_root = RUN_DIR.resolve()
    if not setting_dir.is_relative_to(run_root):
        raise ValueError(f"Setting directory is outside output run: {setting_dir}")
    if setting_dir.exists() and not setting_dir.is_dir():
        raise FileExistsError(f"Setting path is not a directory: {setting_dir}")
    return setting_dir


def _write_setting_settings(setting_dir: Path, settings: dict[str, Any]) -> str:
    settings_fingerprint = fingerprint(settings)
    (setting_dir / "settings.json").write_text(
        json.dumps({"settings": json_value(settings), "fingerprint": settings_fingerprint}, indent=2),
        encoding="utf-8",
    )
    return settings_fingerprint


def _validate_setting_settings(setting_dir: Path, settings: dict[str, Any]) -> str:
    settings_path = setting_dir / "settings.json"
    if not settings_path.is_file():
        raise RuntimeError(f"Non-empty setting directory has no settings.json or completion marker: {setting_dir}")
    try:
        saved = json.loads(settings_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise RuntimeError(f"Cannot read setting settings: {settings_path}") from error
    settings_fingerprint = fingerprint(settings)
    if (
        not isinstance(saved, dict)
        or saved.get("fingerprint") != settings_fingerprint
        or fingerprint(saved.get("settings")) != settings_fingerprint
    ):
        raise RuntimeError(f"Checkpoint settings mismatch: {setting_dir}")
    return settings_fingerprint


def _best_candidates(setting_dir: Path) -> list[Path]:
    return sorted(path.resolve() for path in setting_dir.glob("best*.ckpt") if path.is_file())


def _validated_completion(setting_dir: Path, settings: dict[str, Any], checkpoint: Path) -> Path:
    marker_path = setting_dir / TRAINING_COMPLETE_FILENAME
    if not marker_path.is_file():
        raise RuntimeError(f"Non-empty setting directory has no valid {TRAINING_COMPLETE_FILENAME}: {setting_dir}")
    try:
        marker = json.loads(marker_path.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        raise RuntimeError(f"Cannot read setting completion marker: {marker_path}") from error
    expected_fingerprint = fingerprint(settings)
    expected_checkpoint = checkpoint.resolve()
    expected_sha256 = sha256_file(expected_checkpoint)
    if (
        not isinstance(marker, dict)
        or marker.get("status") != "complete"
        or marker.get("settings_fingerprint") != expected_fingerprint
        or marker.get("checkpoint") != str(expected_checkpoint)
        or marker.get("checkpoint_sha256") != expected_sha256
    ):
        raise RuntimeError(f"Invalid or mismatched {TRAINING_COMPLETE_FILENAME}: {marker_path}")
    return expected_checkpoint


def best_checkpoint(setting_dir: Path, settings: dict[str, Any]) -> Path | None:
    setting_dir = _validate_setting_dir(setting_dir)
    if not setting_dir.exists():
        setting_dir.mkdir(parents=True)
        _write_setting_settings(setting_dir, settings)
        return None
    if not any(setting_dir.iterdir()):
        _write_setting_settings(setting_dir, settings)
        return None
    _validate_setting_settings(setting_dir, settings)
    if not (setting_dir / TRAINING_COMPLETE_FILENAME).is_file():
        raise RuntimeError(f"Non-empty setting directory has no valid {TRAINING_COMPLETE_FILENAME}: {setting_dir}")
    candidates = _best_candidates(setting_dir)
    if len(candidates) != 1:
        raise RuntimeError(f"Expected exactly one best*.ckpt before reuse in setting directory {setting_dir}, found {len(candidates)}")
    return _validated_completion(setting_dir, settings, candidates[0])


def finalize_training(setting_dir: Path, settings: dict[str, Any]) -> Path:
    setting_dir = _validate_setting_dir(setting_dir)
    if not setting_dir.exists():
        raise RuntimeError(f"Cannot finalize training in missing setting directory: {setting_dir}")
    settings_fingerprint = _validate_setting_settings(setting_dir, settings)
    candidates = _best_candidates(setting_dir)
    if len(candidates) != 1:
        raise RuntimeError(f"Training finished without exactly one best*.ckpt in {setting_dir}, found {len(candidates)}")
    checkpoint = candidates[0]
    marker = {
        "status": "complete",
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": sha256_file(checkpoint),
        "settings_fingerprint": settings_fingerprint,
    }
    (setting_dir / TRAINING_COMPLETE_FILENAME).write_text(json.dumps(marker, indent=2), encoding="utf-8")
    return checkpoint


def checkpoint_state(checkpoint: Path) -> dict[str, torch.Tensor]:
    payload = torch.load(checkpoint, map_location="cpu")
    state = payload.get("state_dict")
    if not state:
        raise ValueError(f"Checkpoint has no state_dict: {checkpoint}")
    result = {key.removeprefix("model."): value for key, value in state.items() if key.startswith("model.")}
    if not result:
        raise ValueError(f"Checkpoint has no model state: {checkpoint}")
    return result


def load_gelu_model(checkpoint: Path, dropout: float) -> GELUXASBlock:
    model = GELUXASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, dropout)
    model.load_state_dict(checkpoint_state(checkpoint), strict=True)
    return model.to(DEVICE).eval()


def make_callbacks(setting_dir: Path, monitor: str, patience: int, name: str):
    return [
        EarlyStopping(monitor=monitor, mode="min", patience=patience),
        ModelCheckpoint(
            dirpath=str(setting_dir),
            filename=f"best-{name}-{{epoch:03d}}-{{{monitor}:.8f}}",
            monitor=monitor,
            mode="min",
            save_top_k=1,
            save_last=True,
        ),
    ]


## 6. Train or safely reuse the balanced GELU UniversalXAS model

This is exactly one UniversalXAS run with seed 44. The sampler uses four rows per element in each batch of 32.


In [7]:
universal_dir = RUN_DIR / "universalXAS" / "gelu_seed44"
universal_checkpoint = best_checkpoint(universal_dir, universal_settings)
if universal_checkpoint is None:
    pl.seed_everything(UNIVERSAL_SEED, workers=True)
    universal_module = BalancedUniversalModule(
        val_baselines=val_baselines,
        model=GELUXASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, UNIVERSAL_DROPOUT),
        optimizer=torch.optim.Adam,
        lr=UNIVERSAL_LR,
        lr_scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau,
        lr_scheduler_kwargs={
            "mode": UNIVERSAL_SCHEDULER_MODE,
            "factor": UNIVERSAL_SCHEDULER_FACTOR,
            "patience": effective_scheduler_patience,
            "min_lr": UNIVERSAL_MIN_LR,
        },
        lr_scheduler_interval="epoch",
        lr_scheduler_frequency=UNIVERSAL_SCHEDULER_FREQUENCY,
        lr_scheduler_monitor=UNIVERSAL_MONITOR,
    )
    universal_trainer = pl.Trainer(
        max_epochs=UNIVERSAL_MAX_EPOCHS,
        accelerator="auto",
        devices=1,
        check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
        callbacks=make_callbacks(universal_dir, UNIVERSAL_MONITOR, effective_early_stopping_patience, "universal"),
        logger=CSVLogger(save_dir=str(universal_dir), name="logs"),
        enable_progress_bar=True,
        log_every_n_steps=1,
    )
    universal_trainer.fit(
        universal_module,
        datamodule=UniversalBalancedData(real_sampler),
    )
    universal_checkpoint = finalize_training(universal_dir, universal_settings)
else:
    print(f"Reusing UniversalXAS checkpoint: {universal_checkpoint}")
universal_state = checkpoint_state(universal_checkpoint)


Seed set to 44
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/train_balanced_gelu_universal_tuned_feff/universalXAS/gelu_seed44 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type         | Params | Mode  | FLOPs
-------------------------------------------------------
0 | loss  | MSELoss      | 0      | train | 0    
1 | model | GELUXASBlock | 639 K  | train | 0    
-------------------------------------------------------
639 K     Trainable params
0         Non-train

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |                                                                                       | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

`Trainer.fit` stopped: `max_epochs=800` reached.


## 7. Evaluate the new UniversalXAS model

The fixed Universal configuration is evaluated once on validation and test for every FEFF task. Eta uses that task's training-mean spectrum.


In [8]:
def predict(model: nn.Module, X: np.ndarray) -> np.ndarray:
    loader = DataLoader(TensorDataset(torch.as_tensor(X, dtype=torch.float32)), batch_size=1024, shuffle=False)
    values = []
    model.eval()
    with torch.inference_mode():
        for (batch,) in loader:
            values.append(model(batch.to(DEVICE)).cpu().numpy())
    return np.concatenate(values)


def eta_metrics(split: MLSplits, split_name: str, predictions: np.ndarray) -> dict[str, float]:
    target = getattr(split, split_name).y
    train_mean = split.train.y.mean(axis=0, keepdims=True)
    median_mse = float(np.median(np.mean((target - predictions) ** 2, axis=1)))
    baseline_median_mse = float(np.median(np.mean((target - train_mean) ** 2, axis=1)))
    if median_mse <= 0 or baseline_median_mse <= 0:
        raise ValueError(f"Invalid eta MSE for {split_name}: model={median_mse}, baseline={baseline_median_mse}")
    return {
        f"{split_name}_median_mse": median_mse,
        f"{split_name}_baseline_median_mse": baseline_median_mse,
        f"{split_name}_eta": baseline_median_mse / median_mse,
    }


def evaluate_one(checkpoint: Path, model_name: str, dataset: str, dropout: float, extra: dict[str, Any] | None = None) -> dict[str, Any]:
    model = load_gelu_model(checkpoint, dropout)
    split = splits[dataset]
    row: dict[str, Any] = {
        "model": model_name,
        "dataset": dataset,
        "element": dataset.removesuffix("_FEFF"),
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": sha256_file(checkpoint),
    }
    if extra:
        row.update(extra)
    for split_name in ("val", "test"):
        row.update(eta_metrics(split, split_name, predict(model, getattr(split, split_name).X)))
    return row


universal_rows = [
    evaluate_one(
        universal_checkpoint,
        "UniversalXAS",
        dataset,
        UNIVERSAL_DROPOUT,
        {"seed": UNIVERSAL_SEED, "activation": "GELU", "val_balanced_rel_mse": np.nan},
    )
    for dataset in FEFF_DATASETS
]
universal_eval = pd.DataFrame(universal_rows).sort_values("dataset")
universal_eval["val_balanced_rel_mse"] = universal_eval["val_median_mse"] / universal_eval["val_baseline_median_mse"]
universal_eval.to_csv(RUN_DIR / "universal_eval.csv", index=False)
if len(universal_eval) != len(FEFF_DATASETS):
    raise AssertionError("Universal evaluation must contain one row per FEFF task")


## 8. Train or safely reuse one GELU Tuned-UniversalXAS model per FEFF task

Each task starts from the newly trained balanced GELU UniversalXAS state. The prior row supplies hyperparameters only. Each task uses standard shuffled full per-task training data, cosine annealing, validation every two epochs, and `val_median_mse` checkpointing.


In [9]:
def train_tuned(dataset: str, config: dict[str, Any]) -> tuple[Path, dict[str, Any]]:
    setting = config["setting"]
    setting_dir = RUN_DIR / "tunedUniversalXAS" / dataset / setting
    tuned_settings = {
        "model": "Tuned-UniversalXAS",
        "dataset": dataset,
        "activation": "GELU",
        "input_dim": INPUT_DIM,
        "hidden_dims": HIDDEN_DIMS,
        "output_dim": OUTPUT_DIM,
        **config,
        "optimizer": "Adam",
        "scheduler": "CosineAnnealingLR",
        "monitor": TUNED_MONITOR,
        "check_val_every_n_epoch": CHECK_VAL_EVERY_N_EPOCH,
        "initialization": "new balanced GELU UniversalXAS state",
        "initialization_checkpoint": str(universal_checkpoint),
        "prior_validation_source": str(PRIOR_VALIDATION_CSV),
        "prior_selected_row": selected_prior_rows[dataset],
        "prior_weights_loaded": False,
    }
    checkpoint = best_checkpoint(setting_dir, tuned_settings)
    if checkpoint is None:
        pl.seed_everything(int(config["seed"]), workers=True)
        module = PlModule(
            model=GELUXASBlock(INPUT_DIM, HIDDEN_DIMS, OUTPUT_DIM, float(config["dropout"])),
            optimizer=torch.optim.Adam,
            lr=float(config["lr"]),
            lr_scheduler=torch.optim.lr_scheduler.CosineAnnealingLR,
            lr_scheduler_kwargs={"T_max": int(config["cosine_t"]), "eta_min": float(config["eta_min"])},
            lr_scheduler_interval="epoch",
        )
        module.model.load_state_dict(universal_state, strict=True)
        trainer = pl.Trainer(
            max_epochs=int(config["max_epochs"]),
            accelerator="auto",
            devices=1,
            check_val_every_n_epoch=CHECK_VAL_EVERY_N_EPOCH,
            callbacks=make_callbacks(setting_dir, TUNED_MONITOR, int(config["patience"]), f"{dataset}-{setting}"),
            logger=CSVLogger(save_dir=str(setting_dir), name="logs"),
            enable_progress_bar=True,
            log_every_n_steps=1,
        )
        trainer.fit(module, datamodule=StandardTaskData(splits[dataset], int(config["task_batch"])))
        checkpoint = finalize_training(setting_dir, tuned_settings)
    else:
        print(f"Reusing tuned checkpoint for {dataset}: {checkpoint}")
    provenance = {
        "model": "Tuned-UniversalXAS",
        "dataset": dataset,
        "activation": "GELU",
        "prior_validation_source": str(PRIOR_VALIDATION_CSV),
        "prior_selected_row": selected_prior_rows[dataset],
        "selected_settings": config,
        "initialization_checkpoint": str(universal_checkpoint),
        "initialization_checkpoint_sha256": sha256_file(universal_checkpoint),
        "prior_weights_loaded": False,
        "checkpoint": str(checkpoint),
        "checkpoint_sha256": sha256_file(checkpoint),
        "monitor": TUNED_MONITOR,
    }
    (setting_dir / "source_provenance.json").write_text(json.dumps(json_value(provenance), indent=2), encoding="utf-8")
    return checkpoint, tuned_settings


tuned_checkpoints: dict[str, Path] = {}
tuned_setting_records: dict[str, dict[str, Any]] = {}
for dataset in FEFF_DATASETS:
    checkpoint, setting_record = train_tuned(dataset, tuned_configs[dataset])
    tuned_checkpoints[dataset] = checkpoint
    tuned_setting_records[dataset] = setting_record


Seed set to 145
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /mnt/c/Users/anton/desktop/fulltrainingcopy072726/m3gnetAll8E2EUniversal/e2e_universal_seed42/train_balanced_gelu_universal_tuned_feff/tunedUniversalXAS/Ti_FEFF/cosT750_lr3e-4_do0p1_p60 exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type         | Params | Mode  | FLOPs
-------------------------------------------------------
0 | loss  | MSELoss      | 0      | train | 0    
1 | model | GELUXASBlock | 639 K  | train | 0    
-------------------------------------------------------
639 K     Trainable

Sanity Checking: |                                                                                | 0/? [00:00…

/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.
/home/anton/miniconda3/envs/omnixas/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=31` in the `DataLoader` to improve performance.


Training: |                                                                                       | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

...

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

Validation: |                                                                                     | 0/? [00:00…

## 9. Evaluate tuned models, compare after evaluation, and write completion files

The new tuned validation and test eta values are calculated once per task. Only after all new evaluation completes does this cell read prior Universal and tuned metric files for the comparison. The comparison cannot affect training or selection.


In [10]:
tuned_rows = []
for dataset in FEFF_DATASETS:
    config = tuned_configs[dataset]
    tuned_rows.append(
        evaluate_one(
            tuned_checkpoints[dataset],
            "Tuned-UniversalXAS",
            dataset,
            float(config["dropout"]),
            {
                "setting": config["setting"],
                "seed": config["seed"],
                "activation": "GELU",
                "lr": config["lr"],
                "cosine_t": config["cosine_t"],
                "eta_min": config["eta_min"],
                "max_epochs": config["max_epochs"],
                "patience": config["patience"],
                "task_batch": config["task_batch"],
                "prior_validation_source": str(PRIOR_VALIDATION_CSV),
            },
        )
    )
tuned_eval = pd.DataFrame(tuned_rows).sort_values("dataset")
tuned_eval.to_csv(RUN_DIR / "tuned_eval.csv", index=False)
if len(tuned_eval) != len(FEFF_DATASETS):
    raise AssertionError("Tuned evaluation must contain one row per FEFF task")

# Read prior metric files only after all new training and evaluation.
prior_universal_csv = PRIOR_RUN / "analysis" / "universal_selected_by_dataset.csv"
prior_tuned_csv = PRIOR_RUN / "analysis" / "tuned_eval.csv"
for path in (prior_universal_csv, prior_tuned_csv):
    if not path.is_file():
        raise FileNotFoundError(f"Missing prior metric file for post-evaluation comparison: {path}")
prior_universal = pd.read_csv(prior_universal_csv)
prior_tuned = pd.read_csv(prior_tuned_csv)
if len(prior_universal) != len(FEFF_DATASETS) or len(prior_tuned) != len(FEFF_DATASETS):
    raise ValueError("Prior metric files must contain one selected row per FEFF task")
if set(prior_universal["dataset"]) != set(FEFF_DATASETS) or set(prior_tuned["dataset"]) != set(FEFF_DATASETS):
    raise ValueError("Prior metric files must contain exactly the eight FEFF tasks")
comparison_rows = []
for model_name, new_frame, prior_frame in (
    ("UniversalXAS", universal_eval, prior_universal),
    ("Tuned-UniversalXAS", tuned_eval, prior_tuned),
):
    for dataset in FEFF_DATASETS:
        new_row = new_frame[new_frame["dataset"] == dataset]
        old_row = prior_frame[prior_frame["dataset"] == dataset]
        if len(new_row) != 1 or len(old_row) != 1:
            raise ValueError(f"Comparison requires one new and one prior row for {model_name} {dataset}")
        new_row, old_row = new_row.iloc[0], old_row.iloc[0]
        old_val = "selected_val_eta" if model_name == "UniversalXAS" else "val_eta"
        old_test = "selected_test_eta" if model_name == "UniversalXAS" else "test_eta"
        comparison_rows.append(
            {
                "model": model_name,
                "dataset": dataset,
                "new_checkpoint": new_row["checkpoint"],
                "new_checkpoint_sha256": new_row["checkpoint_sha256"],
                "new_val_eta": new_row["val_eta"],
                "new_test_eta": new_row["test_eta"],
                "prior_checkpoint": old_row.get("selected_universal_checkpoint", old_row.get("checkpoint", "")),
                "prior_val_eta": old_row[old_val],
                "prior_test_eta": old_row[old_test],
                "delta_val_eta": new_row["val_eta"] - old_row[old_val],
                "delta_test_eta": new_row["test_eta"] - old_row[old_test],
                "prior_metrics_source": str(prior_universal_csv if model_name == "UniversalXAS" else prior_tuned_csv),
            }
        )
comparison = pd.DataFrame(comparison_rows).sort_values(["model", "dataset"])
comparison.to_csv(RUN_DIR / "comparison_vs_prior_balanced.csv", index=False)

manifest.update(
    {
        "status": "complete",
        "feature_provenance": "fixed exported 64D features from OMNIXAS_E2E_UNIVERSAL_RUN",
        "universal_checkpoint": str(universal_checkpoint),
        "universal_checkpoint_sha256": sha256_file(universal_checkpoint),
        "tuned_checkpoints": [
            {"dataset": dataset, "checkpoint": str(tuned_checkpoints[dataset]), "checkpoint_sha256": sha256_file(tuned_checkpoints[dataset]), "settings": tuned_configs[dataset]}
            for dataset in FEFF_DATASETS
        ],
        "evaluation_files": {
            "universal": str(RUN_DIR / "universal_eval.csv"),
            "tuned": str(RUN_DIR / "tuned_eval.csv"),
            "comparison": str(RUN_DIR / "comparison_vs_prior_balanced.csv"),
        },
        "checkpoint_hashes_recorded": True,
    }
)
manifest_path.write_text(json.dumps(json_value(manifest), indent=2), encoding="utf-8")
run_complete = {
    "status": "complete",
    "completion_label": "balanced_gelu_universal_tuned_feff_complete",
    "settings_manifest": str(manifest_path),
    "universal_checkpoint": str(universal_checkpoint),
    "universal_checkpoint_sha256": sha256_file(universal_checkpoint),
    "tuned_checkpoints": manifest["tuned_checkpoints"],
    "evaluation_files": manifest["evaluation_files"],
    "selection_policy": "Prior validation-only max val_eta per FEFF dataset. Test eta never selects a setting.",
}
(RUN_DIR / "RUN_COMPLETE.json").write_text(json.dumps(json_value(run_complete), indent=2), encoding="utf-8")
print(comparison[["model", "dataset", "new_val_eta", "prior_val_eta", "new_test_eta", "prior_test_eta"]].to_string(index=False))


             model dataset  new_val_eta  prior_val_eta  new_test_eta  prior_test_eta
Tuned-UniversalXAS Co_FEFF    32.440957      32.099913     29.849538       29.676256
Tuned-UniversalXAS Cr_FEFF    15.947507      15.194700     18.311791       16.083803
Tuned-UniversalXAS Cu_FEFF     7.080356       6.901340      6.653764        7.013738
Tuned-UniversalXAS Fe_FEFF    15.091674      15.406490     14.528522       15.165933
Tuned-UniversalXAS Mn_FEFF    38.223011      39.466672     32.653743       33.606713
Tuned-UniversalXAS Ni_FEFF    18.282290      17.888856     16.879924       16.676424
Tuned-UniversalXAS Ti_FEFF    11.123647      10.966384     11.736147       11.525022
Tuned-UniversalXAS  V_FEFF    13.890307      14.124682     11.990160       12.183384
      UniversalXAS Co_FEFF    27.344196      27.266865     27.133866       26.144838
      UniversalXAS Cr_FEFF    15.228841      14.799168     18.498384       17.700673
      UniversalXAS Cu_FEFF     6.724983       6.510902      6.778